In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load and prepare dataset
dataset = load_dataset("roneneldan/TinyStories")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Add special tokens for chat format
special_tokens = {
    "pad_token": "<PAD>",
    "bos_token": "<BOS>",
    "eos_token": "<EOS>",
    "unk_token": "<UNK>",
    "sep_token": "<SEP>"
}
tokenizer.add_special_tokens(special_tokens)


c:\Users\vbdon\Documents\tinyLLM\code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\vbdon\Documents\tinyLLM\code\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vbdon\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In ord

5

In [2]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformernew import MiniGPT
from torch import nn
from dataloaders import get_data_splits
from torch.utils.data import DataLoader
from utils import collate_fn_fixed_length
import torch 


class TrainingConfig:
    # Model parameters
    vocab_size = 50000
    d_model = 512
    num_heads = 8
    num_layers = 6
    max_seq_len = 1024
    
    # Training parameters
    batch_size = 8  # Adjust based on GPU memory
    learning_rate = 5e-4
    weight_decay = 0.01
    max_epochs = 10
    warmup_steps = 1000
    gradient_clip_val = 1.0


c:\Users\vbdon\Documents\tinyLLM\code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

config = TrainingConfig()

# Initialize model
model = MiniGPT(
    vocab_size=config.vocab_size,
    d_model=config.d_model,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    max_seq_len=config.max_seq_len
)
# --- 1. Determine the device (CPU or GPU) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# --- 2. Move the model to the chosen device ---
model.to(device)

# Setup training components
optimizer = optim.AdamW(
    model.parameters(), 
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

scheduler = CosineAnnealingLR(optimizer, T_max=config.max_epochs)
criterion = nn.CrossEntropyLoss()

# Training loop


train_ds, eval_ds, tokenizer = get_data_splits()
batch_size = 32  # tune to your GPU memory

train_dataloader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn_fixed_length,
    num_workers=4,
    pin_memory=False
)

eval_dataloader = None
if eval_ds is not None:
    eval_dataloader = DataLoader(
        eval_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn_fixed_length,
        num_workers=4,
        pin_memory=False
    )



In [ ]:
train_ds['input_ids']

TypeError: Sequence.count() missing 1 required positional argument: 'value'

In [9]:
tokenizer.vocab.keys()

dict_keys(['Ġnegotiations', 'ĠStra', 'virt', 'kill', 'senal', 'ĠAside', 'strom', 'Ġfused', 'plates', 'ĠTAG', 'Ġalign', 'Ġadolesc', 'Ġinvolve', 'Ġdiplomats', 'Ne', 'Ġtearing', 'Ġlou', 'fle', 'bourne', 'rafted', 'ĠSavannah', 'Uber', 'bring', 'Florida', 'ĠAppendix', 'ĠLocated', 'eed', 'conference', 'Ġtyrant', 'ĠKeith', 'Def', 'Ġtough', 'plays', 'Ġprobability', 'Ġsharper', 'Ġeuph', 'ĠTWO', 'ĠThough', 'emo', 'Ġtrademarks', 'ĠGrav', 'ĠFile', 'ĠCFR', 'Ġpersonality', 'ĠRussian', 'ĠTok', 'TA', 'ĠDenis', 'ĠguiIcon', 'ĠAshley', 'chair', 'ĠSocialism', 'Ġinstitution', 'Ġconstitution', 'Ġoccup', 'ĠBurton', 'Ġdaemon', 'Ġquestioning', 'unk', 'ĠDominion', 'erning', 'Ts', 'ĠDiver', 'Ġeats', 'ĠShad', 'Serv', 'SA', 'missible', 'Ġstarting', 'ĠPER', 'essing', 'iovascular', 'Ġkept', 'othermal', 'ĠAntioch', 'Ġloopholes', 'ild', '727', 'Ġexecut', 'AME', 'Ġfeas', 'Ġservices', 'ĠWinter', 'balance', 'Ġwires', 'Ġisland', 'Serial', 'Ġretiring', 'Ġreorgan', 'groups', 'Food', 'lene', 'Standard', 'Ġfant', 'Ġexecutable

In [ ]:
model.train()
for epoch in range(config.max_epochs):
    total_loss = 0
    for batch_idx, batch in enumerate(train_dataloader):
        input_ids = batch['input_ids'].to(device)
        labels = input_ids[:, 1:].contiguous()  # Shift for next-token prediction
        input_ids = input_ids[:, :-1].contiguous()

        # Forward pass
        logits = model(input_ids)
        loss = criterion(logits.view(-1, config.vocab_size), labels.view(-1))

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_val)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")

    scheduler.step()
    print(f"Epoch {epoch} completed. Average loss: {total_loss/len(train_dataloader):.4f}")

In [ ]:
# Format training data for instruction following
def format_chat_data(examples):
    formatted_texts = []
    for conversation in examples['conversations']:
        text = ""
        for turn in conversation:
            if turn['role'] == 'user':
                text += f"<|user|>{turn['content']}<|end|>\n"
            elif turn['role'] == 'assistant':
                text += f"<|assistant|>{turn['content']}<|end|>\n"
        formatted_texts.append(text)
    return {'text': formatted_texts}

# Load instruction dataset
instruction_dataset = load_dataset("Open-Orca/OpenOrca")
instruction_dataset = instruction_dataset.map(format_chat_data, batched=True)

# Fine-tune with lower learning rate
finetune_optimizer = optim.AdamW(model.parameters(), lr=1e-5)
